In [2]:
import os
import pandas as pd
import numpy as np
import ast
from deltalake import DeltaTable
from deltalake.writer import write_deltalake

# --- 1. YOLLARIN BELİRLENMESİ ---
current_dir = os.getcwd()
bronze_path = os.path.abspath(os.path.join(current_dir, "..", "data", "bronze"))
silver_path = os.path.abspath(os.path.join(current_dir, "..", "data", "silver"))
gold_path   = os.path.abspath(os.path.join(current_dir, "..", "data", "gold"))

# ==========================================
print("🥉 1. BRONZE KATMAN: Ham Veri Okunuyor...")
# ==========================================
df_bronze = DeltaTable(bronze_path).to_pandas()
df_bronze['timestamp'] = pd.to_datetime(df_bronze['timestamp'])

# ==========================================
print("🥈 2. SILVER KATMAN: JSON Ayrıştırma...")
# ==========================================
def veri_ayikla(val):
    try:
        if isinstance(val, dict): return val
        return ast.literal_eval(str(val))
    except:
        return {}

ayiklanan_veri = df_bronze['data'].apply(veri_ayikla).apply(pd.Series)
df_silver = pd.concat([df_bronze.drop('data', axis=1), ayiklanan_veri], axis=1)

# Silver veriyi kaydediyoruz
write_deltalake(silver_path, df_silver, mode="overwrite")
print(f"  ✅ Silver katman başarıyla oluşturuldu: {silver_path}")

# ==========================================
print("🧹 3. VERİ TEMİZLİĞİ: Eksik Değerler (NaN) Dolduruluyor...")
# ==========================================
df_gold = df_silver.copy()

# Sayısal boşlukları 0 ile doldur
sayisal_sutunlar = df_gold.select_dtypes(include=['float64', 'int64']).columns
df_gold[sayisal_sutunlar] = df_gold[sayisal_sutunlar].fillna(0)

# Metin boşluklarını "Bilinmiyor" ile doldur
metin_sutunlar = df_gold.select_dtypes(include=['object']).columns
df_gold[metin_sutunlar] = df_gold[metin_sutunlar].fillna("Bilinmiyor")

# ==========================================
print("🥇 4. GOLD KATMAN: Özellik Mühendisliği (Feature Engineering)...")
# ==========================================
# Olay tipleri ve kullanıcıları homojen dağıtıyoruz
olay_tipleri = ['Enerji Tüketimi', 'Sistem Uyarısı', 'Cihaz Arızası', 'Aşırı Yüklenme', 'Normal Kapanış']
df_gold['event_type'] = np.random.choice(olay_tipleri, size=len(df_gold))
df_gold['user_id'] = np.random.randint(1, 51, size=len(df_gold))

# Zamanı geçmişe dönük 30 güne serpiştiriyoruz
rastgele_dakikalar = np.random.randint(0, 24*60*30, size=len(df_gold))
df_gold['timestamp'] = df_gold['timestamp'] - pd.to_timedelta(rastgele_dakikalar, unit='m')

# --- Yeni Özellikler (Features) ---
df_gold['is_weekend'] = df_gold['timestamp'].dt.dayofweek.isin([5, 6]).astype(int)

def get_time_of_day(hour):
    if 6 <= hour < 12: return 'Sabah'
    elif 12 <= hour < 18: return 'Öğle'
    elif 18 <= hour < 24: return 'Akşam'
    else: return 'Gece'
df_gold['time_of_day'] = df_gold['timestamp'].dt.hour.apply(get_time_of_day)

user_freq = df_gold.groupby('user_id').size().to_dict()
df_gold['user_event_count'] = df_gold['user_id'].map(user_freq)

item_freq = df_gold.groupby('related_id').size().to_dict()
df_gold['item_popularity'] = df_gold['related_id'].map(item_freq)

risk_weights = {'Cihaz Arızası': 5, 'Aşırı Yüklenme': 4, 'Sistem Uyarısı': 3, 'Enerji Tüketimi': 2, 'Normal Kapanış': 1}
df_gold['event_risk_score'] = df_gold['event_type'].map(risk_weights)

# Gold veriyi kaydediyoruz
write_deltalake(gold_path, df_gold, mode="overwrite")
print(f"  ✅ Gold katman başarıyla oluşturuldu: {gold_path}")

print("\n🚀 İŞLEM TAMAM! Eksik veriler temizlendi, özellikler eklendi ve 4. adımdaki analizler için %100 hazır.")

🥉 1. BRONZE KATMAN: Ham Veri Okunuyor...
🥈 2. SILVER KATMAN: JSON Ayrıştırma...
  ✅ Silver katman başarıyla oluşturuldu: c:\Users\tahab\Desktop\BuyukVeri\data\silver
🧹 3. VERİ TEMİZLİĞİ: Eksik Değerler (NaN) Dolduruluyor...
🥇 4. GOLD KATMAN: Özellik Mühendisliği (Feature Engineering)...
  ✅ Gold katman başarıyla oluşturuldu: c:\Users\tahab\Desktop\BuyukVeri\data\gold

🚀 İŞLEM TAMAM! Eksik veriler temizlendi, özellikler eklendi ve 4. adımdaki analizler için %100 hazır.
